# درس 11 - پروتکل کانتکست مدل (MCP)

پروتکل کانتکست مدل (**MCP**) یک استاندارد باز است که به عامل‌ها امکان می‌دهد در زمان اجرا ابزارها، منابع و منابع داده را به‌صورت پویا کشف و استفاده کنند. به جای کدنویسی ثابت ابزارها در یک عامل، MCP اجازه می‌دهد عامل‌ها به سرورهای خارجی متصل شوند که قابلیت‌ها را بر اساس تقاضا ارائه می‌دهند.

در این درس، خواهید آموخت:
- MCP چیست و چرا برای سیستم‌های عامل مهم است
- معماری کلاینت-سرور MCP چگونه کار می‌کند
- چگونه عامل‌هایی بسازید که از کشف ابزار به سبک MCP استفاده می‌کنند


## راه‌اندازی

**پیش‌نیازها:**
- پروژه Microsoft Foundry با مدل مستقر شده
- اجرای `az login` برای احراز هویت


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## پروتکل متن مدل (MCP) چیست؟

MCP یک روش استاندارد را برای کشف و تعامل عامل‌های هوش مصنوعی با ابزارها و منابع داده خارجی تعریف می‌کند:

- **سرور MCP**: ابزارها، منابع و فرمان‌ها را از طریق یک پروتکل استاندارد ارائه می‌دهد
- **کلاینت MCP**: محیط اجرای عامل که به سرورها متصل شده و قابلیت‌های موجود را کشف می‌کند
- **کشف پویا**: عامل‌ها نیازی به ابزارهای کدنویسی‌شده ندارد — آنها قابلیت‌های در دسترس را در زمان اجرا کشف می‌کنند

این برای ساخت سیستم‌های افزون‌پذیر عامل‌ها بسیار قدرتمند است، جایی که قابلیت‌های جدید بدون نیاز به تغییر کد عامل اضافه می‌شوند.


## نحوه عملکرد MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. عامل (کلاینت MCP) به یک سرور MCP متصل می‌شود
2. سرور پاسخ می‌دهد با فهرستی از ابزارهای موجود و طرح‌های آن‌ها
3. سپس عامل می‌تواند در طول استدلال خود هر ابزار کشف‌شده‌ای را فراخوانی کند
4. نتایج از طریق همان پروتکل بازمی‌گردند


## شبیه‌سازی کشف ابزار MCP

از آنجایی که یک سرور واقعی MCP نیازمند یک فرایند سرور در حال اجرا است، ما الگو را با استفاده از توابع `@tool` نمایش می‌دهیم که شبیه‌سازی می‌کنند چه چیزی را یک سرویس اقامتی متصل به MCP فراهم می‌آورد.

در محیط تولید، این ابزارها به صورت داینامیک از یک سرور MCP کشف می‌شوند نه اینکه به صورت محلی تعریف شوند.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## ساخت یک عامل با ابزارهای سبک MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP در محیط تولید

در یک محیط تولید، MCP الگوهای قدرتمندی را فعال می‌کند:

- **کشف داینامیک ابزار**: عامل‌ها به سرورهای MCP متصل می‌شوند و در زمان اجرا ابزارها را کشف می‌کنند
- **معماری جداشده**: ارائه‌دهندگان ابزار می‌توانند به صورت مستقل از عامل به‌روزرسانی شوند
- **اشتراک‌گذاری بین سازمان‌ها**: تیم‌ها می‌توانند قابلیت‌ها را از طریق سرورهای MCP در دسترس قرار دهند که هر عاملی می‌تواند استفاده کند
- **پشتیبانی از Microsoft Agent Framework**: MAF دارای پشتیبانی داخلی از کلاینت MCP از طریق ادغام `mcp` است

برای استفاده از یک سرور واقعی MCP با MAF، می‌توانید از طریق `hosted_mcp_tool()` یا ادغام کلاینت MCP متصل شوید.

**بیشتر بدانید:**
- [مشخصات MCP](https://modelcontextprotocol.io/)
- [پشتیبانی Microsoft Agent Framework از MCP](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## خلاصه

در این درس، شما آموختید:
- **MCP** یک استاندارد باز برای کشف پویا ابزارها بین نمایندگان و ارائه‌دهندگان ابزار است
- معماری **کلاینت-سرور** به نمایندگان امکان می‌دهد قابلیت‌ها را در زمان اجرا کشف کنند
- MCP امکان ایجاد سیستم‌های نماینده **گسترش‌پذیر و جداشده** را فراهم می‌کند که ابزارها بدون تغییر کد اضافه می‌شوند
- چارچوب Microsoft Agent، **پشتیبانی داخلی از MCP** برای استفاده در تولید فراهم می‌کند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
